In [ ]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    # Python
    random.seed(seed)
    
    # Numpy
    np.random.seed(seed)
    
    # PyTorch (CPU)
    torch.manual_seed(seed)
    
    # PyTorch (GPU)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Determinismo (importante!)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Algumas libs usam isso
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    # Para transformers (às vezes ajuda)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    print(f"Seed definida como {seed}")

# Uso
seed_everything(42)

Seed definida como 42


In [ ]:
import numpy as np
import os
from datetime import datetime
import pandas as pd
from tqdm import tqdm

import datasets
import evaluate

import torch
import torch.nn as nn

from transformers import AutoTokenizer, BertConfig, BertModel, BertPreTrainedModel

from sklearn.metrics import f1_score, accuracy_score

/home/guilhermelima/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Mapeamento real do dataset de treino (ordenado por frequência, não alfabeticamente)
UPOS_LABELS = ['DET', 'NOUN', 'VERB', 'PUNCT', 'SCONJ', 'ADP', 'ADJ',
               'CCONJ', 'ADV', 'PROPN', 'AUX', 'NUM', 'PRON', 'SYM', 'X', 'INTJ']

DEPREL_LABELS = ['det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark',
                 'advcl', 'case', 'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod',
                 'flat:name', 'ccomp', 'cop', 'acl', 'nummod', 'acl:relcl',
                 'ccomp:speech', 'parataxis', 'csubj', 'aux:pass', 'appos', 'fixed',
                 'nsubj:pass', 'aux', 'nsubj:outer', 'obl:agent', 'expl:impers',
                 'expl', 'discourse', 'orphan', 'dislocated', 'flat', 'flat:foreign',
                 'iobj', 'vocative', 'csubj:outer', 'list', 'reparandum', 'csubj:pass']

DEPREL_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(DEPREL_LABELS)
}

IDX_TO_DEPREL_LABELS = {
    i: j
    for j, i in DEPREL_LABELS_TO_IDX.items()
}


UPOS_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(UPOS_LABELS)
}

IDX_TO_UPOS_LABELS = {
    i: j
    for j, i in UPOS_LABELS_TO_IDX.items()
}


NUM_TRAIN_EPOCHS = 10

# ── Para trocar de modelo, altere apenas estas duas variáveis ──────────────────
# BERTimbau-large : FINETUNED_MODEL_PATH = '.../checkpoint-28743'
#                   TOKENIZER_NAME = 'neuralmind/bert-large-portuguese-cased'
# modernJabutica  : FINETUNED_MODEL_PATH = '.../checkpoint-29480'
#                   TOKENIZER_NAME = 'amadeusai/modernJabuticaBERT-Base-1k'
FINETUNED_MODEL_PATH = '/home/guilhermelima/msc/trainer_output/checkpoint-28006'
TOKENIZER_NAME       = 'amadeusai/modernJabuticaBERT-Base-1k'
# ──────────────────────────────────────────────────────────────────────────────


In [ ]:
# model definition
'''class MultiTaskSentencePrediction(BertPreTrainedModel):
    def __init__(self, config, num_xpos_labels, num_deprel_labels):
        super().__init__(config)
        self.num_xpos_labels = num_xpos_labels
        self.num_deprel_labels = num_deprel_labels

        self.bert = BertModel(config)

        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)
        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)

        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        
        self.dropout = nn.Dropout(classifier_dropout)
        self.init_weights()

    def forward(
            self, input_ids, attention_mask=None, token_type_ids=None, 
            xpos_label=None, deprel_label=None 
    ):
        outputs = self.bert(
            input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        sequence_output = self.dropout(outputs[0])  # [batch_size, seq_len, hidden_size]


        
        # Classificação por token
        logits_xpos = self.xpos_classifier(sequence_output)     # [batch_size, seq_len, num_xpos_labels]
        logits_deprel = self.deprel_classifier(sequence_output)
        


        loss = None
        if xpos_label != None: #and deprel_label != None:
            
            
            loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)

            loss = loss_fct1(
            logits_xpos.view(-1, self.num_xpos_labels),  # [B * L, num_xpos_labels]
            xpos_label.view(-1)                          # [B * L]
        ) + loss_fct2(
            logits_deprel.view(-1, self.num_deprel_labels),
            deprel_label.view(-1)
        )
        #return (loss, logits_xpos) if loss is not None else (logits_xpos)
        return (loss, logits_xpos, logits_deprel) if loss is not None else (logits_xpos, logits_deprel)'''

'class MultiTaskSentencePrediction(BertPreTrainedModel):\n    def __init__(self, config, num_xpos_labels, num_deprel_labels):\n        super().__init__(config)\n        self.num_xpos_labels = num_xpos_labels\n        self.num_deprel_labels = num_deprel_labels\n\n        self.bert = BertModel(config)\n\n        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)\n        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)\n\n        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob\n        \n        self.dropout = nn.Dropout(classifier_dropout)\n        self.init_weights()\n\n    def forward(\n            self, input_ids, attention_mask=None, token_type_ids=None, \n            xpos_label=None, deprel_label=None \n    ):\n        outputs = self.bert(\n            input_ids=input_ids, attention_mask=attention_mask,\n            token_type_ids=token_type_ids\n        )\n        se

In [ ]:
# model definition
'''class MultiTaskSentencePrediction(BertPreTrainedModel):
    def __init__(self, config, num_xpos_labels, num_deprel_labels, num_upos_labels):
        super().__init__(config)
        self.num_xpos_labels = num_xpos_labels
        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels = num_upos_labels

        self.bert = BertModel(config)

        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)
        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)

        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        
        self.dropout = nn.Dropout(classifier_dropout)
        self.init_weights()

    def forward(
            self, input_ids, attention_mask=None, token_type_ids=None, 
            xpos_label=None, deprel_label=None, upos_label=None 
    ):
        outputs = self.bert(
            input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        sequence_output = self.dropout(outputs[0])  # [batch_size, seq_len, hidden_size]


        
        # Classificação por token
        logits_xpos = self.xpos_classifier(sequence_output)     # [batch_size, seq_len, num_xpos_labels]
        logits_deprel = self.deprel_classifier(sequence_output)
        logits_upos = self.upos_classifier(sequence_output)


        loss = None
        if xpos_label != None and deprel_label != None and upos_label != None:
            
            
            loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct3 = nn.CrossEntropyLoss(ignore_index=-100)

            loss = loss_fct1(
            logits_xpos.view(-1, self.num_xpos_labels),  # [B * L, num_xpos_labels]
            xpos_label.view(-1)                          # [B * L]
        ) + loss_fct2(
            logits_deprel.view(-1, self.num_deprel_labels),
            deprel_label.view(-1)
        ) + loss_fct3(
            logits_upos.view(-1, self.num_upos_labels),
            upos_label.view(-1)
        )

        return (loss, logits_xpos, logits_deprel, logits_upos) if loss is not None else (logits_xpos, logits_deprel, logits_upos)'''

'class MultiTaskSentencePrediction(BertPreTrainedModel):\n    def __init__(self, config, num_xpos_labels, num_deprel_labels, num_upos_labels):\n        super().__init__(config)\n        self.num_xpos_labels = num_xpos_labels\n        self.num_deprel_labels = num_deprel_labels\n        self.num_upos_labels = num_upos_labels\n\n        self.bert = BertModel(config)\n\n        self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)\n        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)\n        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)\n\n        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob\n        \n        self.dropout = nn.Dropout(classifier_dropout)\n        self.init_weights()\n\n    def forward(\n            self, input_ids, attention_mask=None, token_type_ids=None, \n            xpos_label=None, deprel_label=None, upos_label=None \n    ):

In [ ]:
from transformers import PreTrainedModel, AutoModel
# ENCONDER
class MultiTaskSentencePredictionEncoder(PreTrainedModel):
        _tied_weights_keys = []
        all_tied_weights_keys = {}
        def __init__(self, config, num_deprel_labels, num_upos_labels, num_head_labels=200):
            super().__init__(config)
            #self.num_xpos_labels = num_xpos_labels
            
            self.num_deprel_labels = num_deprel_labels
            self.num_upos_labels = num_upos_labels
            self.num_head_labels = num_head_labels
            
            if False:
                self.bert = BertModel(config)
            else:
                self.bert = AutoModel.from_config(config)

            #self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)
            self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
            self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
            self.head_classifier = nn.Linear(config.hidden_size, num_head_labels)
            
            #self.head_classifier = Biaffine(in_features=config.hidden_size, out_features=1)

            classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
            
            self.dropout = nn.Dropout(classifier_dropout)
            self.init_weights()

        def forward(
                self, input_ids, attention_mask=None, token_type_ids=None, 
                deprel_label=None, upos_label=None , head_label=None
        ):
            outputs = self.bert(
                input_ids=input_ids, attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            sequence_output = self.dropout(outputs[0])  # [batch_size, seq_len, hidden_size]
            """
            def debug_labels(name, labels, num_classes):
                    print(f"\n{name}")
                    print("min:", labels.min().item())
                    print("max:", labels.max().item())
                    print("unique:", torch.unique(labels))

                    invalid = (labels >= num_classes) | (labels < -100)
                    if invalid.any():
                        print("❌ VALORES INVÁLIDOS ENCONTRADOS!")

                            # dentro do forward
            debug_labels("deprel", deprel_label, self.num_deprel_labels)
            debug_labels("upos", upos_label, self.num_upos_labels)
            debug_labels("head", head_label, self.num_head_labels)"""
            
            # Classificação por token
            #logits_xpos = self.xpos_classifier(sequence_output)     # [batch_size, seq_len, num_xpos_labels]
            logits_deprel = self.deprel_classifier(sequence_output)
            logits_upos = self.upos_classifier(sequence_output)
            logits_head = self.head_classifier(sequence_output)

            loss = None
            if deprel_label is not None and upos_label is not None and head_label is not None:
                
                
                loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
                loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)
                loss_fct3 = nn.CrossEntropyLoss(ignore_index=-100)
                #loss_fct4 = nn.CrossEntropyLoss(ignore_index=-100)

                loss = loss_fct1(
                logits_deprel.view(-1, self.num_deprel_labels),  # [B * L, num_xpos_labels]
                deprel_label.view(-1)                          # [B * L]
            ) + loss_fct2(
                logits_upos.view(-1, self.num_upos_labels),
                upos_label.view(-1)
            ) + loss_fct3(
                logits_head.view(-1, self.num_head_labels),
                head_label.view(-1)
            )

            return (loss, logits_deprel, logits_upos, logits_head) if loss is not None else (logits_deprel, logits_upos, logits_head)


In [ ]:
from transformers import AutoConfig

# Config sempre carregado do checkpoint: garante arquitetura correta
# (BERT, ModernBERT, mBERT, etc.) independente do modelo base.
MODEL_CONFIG = AutoConfig.from_pretrained(FINETUNED_MODEL_PATH)
TOKENIZER    = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print(f'Arquitetura: {MODEL_CONFIG.model_type} | hidden_size: {MODEL_CONFIG.hidden_size}')
print(f'Tokenizer  : {TOKENIZER_NAME}')


Arquitetura: modernbert | hidden_size: 768
Tokenizer  : amadeusai/modernJabuticaBERT-Base-1k


In [ ]:
# loading model 
model = MultiTaskSentencePredictionEncoder.from_pretrained(
    FINETUNED_MODEL_PATH,
    config=MODEL_CONFIG,
    num_deprel_labels=len(DEPREL_LABELS), num_upos_labels=len(UPOS_LABELS)
).to("cuda" if torch.cuda.is_available() else "cpu")
print('Model loaded successfully...')

Loading weights: 100%|██████████| 140/140 [00:00<00:00, 12559.14it/s]


Model loaded successfully...


In [ ]:
print(len(DEPREL_LABELS))

44


In [ ]:
print(len(UPOS_LABELS))

16


In [ ]:
from transformers import AutoConfig

In [ ]:
#config = AutoConfig.from_pretrained(FINETUNED_MODEL_PATH)

"""model = MultiTaskSentencePredictionEncoder.from_pretrained(
            FINETUNED_MODEL_PATH,
            config=config,
            num_deprel_labels=len(DEPREL_LABELS),
            num_upos_labels=len(UPOS_LABELS)
        ).to("cuda" if torch.cuda.is_available() else "cpu")
"""

'model = MultiTaskSentencePredictionEncoder.from_pretrained(\n            FINETUNED_MODEL_PATH,\n            config=config,\n            num_deprel_labels=len(DEPREL_LABELS),\n            num_upos_labels=len(UPOS_LABELS)\n        ).to("cuda" if torch.cuda.is_available() else "cpu")\n'

In [ ]:
'''import pandas as pd

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    predictions_xpos = []
    predictions_deprel = []
    probability_xpos = []
    probability_deprel = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):  # sentences é list[list[str]]
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        logits_xpos = model_outputs[0]
        logits_deprel = model_outputs[1]

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_xpos, sent_preds_deprel = [], []
        sent_probs_xpos, sent_probs_deprel = [], []

        for word_idx in range(len(tokens)):
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == word_idx]
            if not subtoken_idxs:
                continue

            first_sub = subtoken_idxs[0]

            prob_xpos = torch.softmax(logits_xpos[0, first_sub], dim=-1)
            prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)

            pred_xpos = torch.argmax(prob_xpos).item()
            pred_deprel = torch.argmax(prob_deprel).item()

            sent_preds_xpos.append(IDX_TO_XPOS_LABELS[pred_xpos])
            sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
            sent_probs_xpos.append(prob_xpos[pred_xpos].item())
            sent_probs_deprel.append(prob_deprel[pred_deprel].item())

        predictions_xpos.append(sent_preds_xpos)
        predictions_deprel.append(sent_preds_deprel)
        probability_xpos.append(sent_probs_xpos)
        probability_deprel.append(sent_probs_deprel)

    # monta DataFrame no final
    df_out = pd.DataFrame({
        "tokens": sentences,
        "xpos_predictions": predictions_xpos,
        "xpos_pred_probability": probability_xpos,
        "deprel_predictions": predictions_deprel,
        "deprel_pred_probability": probability_deprel
    })
    return df_out
'''

'import pandas as pd\n\ndef get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):\n    predictions_xpos = []\n    predictions_deprel = []\n    probability_xpos = []\n    probability_deprel = []\n\n    model.eval()\n    model.to(device)\n\n    for tokens in tqdm(sentences):  # sentences é list[list[str]]\n        inputs = tokenizer(\n            tokens,\n            is_split_into_words=True,\n            return_tensors="pt",\n            padding=True,\n            truncation=True\n        ).to(device)\n\n        with torch.no_grad():\n            model_outputs = model(**inputs)\n\n        logits_xpos = model_outputs[0]\n        logits_deprel = model_outputs[1]\n\n        word_ids = inputs.word_ids(batch_index=0)\n\n        sent_preds_xpos, sent_preds_deprel = [], []\n        sent_probs_xpos, sent_probs_deprel = [], []\n\n        for word_idx in range(len(tokens)):\n            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == word_idx]\n            if n

In [ ]:
'''import pandas as pd
import torch
from tqdm import tqdm

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    predictions_xpos = []
    predictions_deprel = []
    predictions_upos = []

    probability_xpos = []
    probability_deprel = []
    probability_upos = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        #print(tokens)
        #print(model_outputs[:-1][-1][0, 0].detach().numpy())

        logits_xpos = model_outputs[0]  # cabeça 0
        logits_deprel = model_outputs[1]  # cabeça 1
        logits_upos = model_outputs[2] # cabeça 2

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_xpos, sent_preds_deprel, sent_preds_upos = [], [], []
        sent_probs_xpos, sent_probs_deprel, sent_probs_upos = [], [], []

        for token_idx in range(len(tokens)):
            # todos os subtokens do token
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == token_idx]

            if subtoken_idxs:
                first_sub = subtoken_idxs[0]  # ou média/max dos subtokens
                prob_xpos = torch.softmax(logits_xpos[0, first_sub], dim=-1)
                prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)
                prob_upos = torch.softmax(logits_upos[0, first_sub], dim=-1)

                pred_xpos = torch.argmax(prob_xpos).item()
                pred_deprel = torch.argmax(prob_deprel).item()
                pred_upos = torch.argmax(prob_upos).item()

                sent_preds_xpos.append(IDX_TO_XPOS_LABELS[pred_xpos])
                sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
                sent_preds_upos.append(IDX_TO_UPOS_LABELS[pred_upos])

                sent_probs_xpos.append(prob_xpos[pred_xpos].item())
                sent_probs_deprel.append(prob_deprel[pred_deprel].item())
                sent_probs_upos.append(prob_upos[pred_upos].item())

            else:
                # mantém correspondência de tamanho com token original
                sent_preds_xpos.append(None)
                sent_preds_deprel.append(None)
                sent_preds_upos.append(None)
                
                sent_probs_xpos.append(None)
                sent_probs_deprel.append(None)
                sent_probs_upos.append(None)

        predictions_xpos.append(sent_preds_xpos)
        predictions_deprel.append(sent_preds_deprel)
        predictions_upos.append(sent_preds_upos)

        probability_xpos.append(sent_probs_xpos)
        probability_deprel.append(sent_probs_deprel)
        probability_upos.append(sent_probs_upos)

    # monta DataFrame
    df_out = pd.DataFrame({
        "tokens": sentences,
        "xpos_predictions": predictions_xpos,
        "xpos_pred_probability": probability_xpos,
        "deprel_predictions": predictions_deprel,
        "deprel_pred_probability": probability_deprel,
        "upos_predictions": predictions_upos,
        "upos_pred_probability": probability_upos       
    })

    return df_out'''


'import pandas as pd\nimport torch\nfrom tqdm import tqdm\n\ndef get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):\n    predictions_xpos = []\n    predictions_deprel = []\n    predictions_upos = []\n\n    probability_xpos = []\n    probability_deprel = []\n    probability_upos = []\n\n    model.eval()\n    model.to(device)\n\n    for tokens in tqdm(sentences):\n        inputs = tokenizer(\n            tokens,\n            is_split_into_words=True,\n            return_tensors="pt",\n            padding=True,\n            truncation=True\n        ).to(device)\n\n        with torch.no_grad():\n            model_outputs = model(**inputs)\n\n        #print(tokens)\n        #print(model_outputs[:-1][-1][0, 0].detach().numpy())\n\n        logits_xpos = model_outputs[0]  # cabeça 0\n        logits_deprel = model_outputs[1]  # cabeça 1\n        logits_upos = model_outputs[2] # cabeça 2\n\n        word_ids = inputs.word_ids(batch_index=0)\n\n        sent_preds_xpos, sent_p

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    #predictions_xpos = []
    predictions_deprel = []
    predictions_upos = []
    predictions_head = []

    #probability_xpos = []
    probability_deprel = []
    probability_upos = []
    probability_head = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        #print(tokens)
        #print(model_outputs[:-1][-1][0, 0].detach().numpy())

        #logits_xpos = model_outputs[0]  # cabeça 0
        logits_deprel = model_outputs[0]  # cabeça 1
        logits_upos = model_outputs[1] # cabeça 2
        logits_head = model_outputs[2] # cabeça 3

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_xpos, sent_preds_deprel, sent_preds_upos, sent_preds_head = [], [], [], []
        sent_probs_xpos, sent_probs_deprel, sent_probs_upos, sent_probs_head = [], [], [], []

        for token_idx in range(len(tokens)):
            # todos os subtokens do token
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == token_idx]

            if subtoken_idxs:
                first_sub = subtoken_idxs[0]  # ou média/max dos subtokens
                #prob_xpos = torch.softmax(logits_xpos[0, first_sub], dim=-1)
                prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)
                prob_upos = torch.softmax(logits_upos[0, first_sub], dim=-1)
                prob_head = torch.softmax(logits_head[0, first_sub], dim=-1)

                #pred_xpos = torch.argmax(prob_xpos).item()
                pred_deprel = torch.argmax(prob_deprel).item()
                pred_upos = torch.argmax(prob_upos).item()
                pred_head = torch.argmax(prob_head).item()

                

                #sent_preds_xpos.append(IDX_TO_XPOS_LABELS[pred_xpos])
                sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
                sent_preds_upos.append(IDX_TO_UPOS_LABELS[pred_upos])
                sent_preds_head.append(pred_head)

                #sent_probs_xpos.append(prob_xpos[pred_xpos].item())
                sent_probs_deprel.append(prob_deprel[pred_deprel].item())
                sent_probs_upos.append(prob_upos[pred_upos].item())
                sent_probs_head.append(prob_head)

            else:
                # mantém correspondência de tamanho com token original
                #sent_preds_xpos.append(None)
                sent_preds_deprel.append(None)
                sent_preds_upos.append(None)
                sent_preds_head.append(None)

                #sent_probs_xpos.append(None)
                sent_probs_deprel.append(None)
                sent_probs_upos.append(None)
                sent_probs_head.append(None)

        #predictions_xpos.append(sent_preds_xpos)
        predictions_deprel.append(sent_preds_deprel)
        predictions_upos.append(sent_preds_upos)
        predictions_head.append(sent_preds_head)

        #probability_xpos.append(sent_probs_xpos)
        probability_deprel.append(sent_probs_deprel)
        probability_upos.append(sent_probs_upos)
        probability_head.append(sent_probs_head)

    # monta DataFrame
    df_out = pd.DataFrame({
        "tokens": sentences,
        #"xpos_predictions": predictions_xpos,
        #"xpos_pred_probability": probability_xpos,
        "deprel_predictions": predictions_deprel,
        "deprel_pred_probability": probability_deprel,
        "upos_predictions": predictions_upos,
        "upos_pred_probability": probability_upos,
        "head_predictions": predictions_head,
        "head_pred_probability": probability_head    
    })

    return df_out


In [ ]:
from datasets import load_from_disk

dataset = load_from_disk('/home/guilhermelima/msc/data_dois/complaints_dataset_obj_outxpos')
test_dataset = dataset['test']
print(f"Test set: {len(test_dataset)} sentenças")

Test set: 1683 sentenças


In [ ]:
test_sentences = test_dataset['tokens']
test_upos      = test_dataset['upos']
test_deprel    = test_dataset['deprel']
test_head      = test_dataset['head_tags']

print(f"Sentenças: {len(test_sentences)}")
print(f"Exemplo tokens : {test_sentences[0]}")
print(f"Exemplo upos   : {test_upos[0]}")
print(f"Exemplo deprel : {test_deprel[0]}")
print(f"Exemplo heads  : {test_head[0]}")

Sentenças: 1683
Exemplo tokens : ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']
Exemplo upos   : ['DET', 'PROPN', 'PROPN', 'ADV', 'VERB', 'DET', 'NOUN', 'PUNCT']
Exemplo deprel : ['det', 'nsubj', 'flat:name', 'advmod', 'root', 'det', 'obj', 'punct']
Exemplo heads  : [2, 5, 2, 5, 0, 7, 5, 5]


In [ ]:
# Células de geração de .conll ignoradas — dados carregados diretamente do dataset de treino
# (test.csv usava anotações UD v1 incompatíveis com o vocabulário do modelo)


In [ ]:
#print(data)

In [ ]:
#def aling_word_for_sentence(df):

#    word_vector = []

#    for taggings in df:
#        word_vector.append(" ".join(taggings))
#    return word_vector

#test_df['sentence'] = aling_word_for_sentence(test_df['tokens'])

In [ ]:
#!pip install conllu

In [ ]:
# test_sentences, test_upos, test_deprel, test_head já definidos na célula anterior
print(f"Total de sentenças de teste: {len(test_sentences)}")
print(f"Primeira sentença: {test_sentences[0]}")

Total de sentenças de teste: 1683
Primeira sentença: ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [ ]:
test_sentences

Column([['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.'], ['A', 'Odebrecht', 'pagou', '300', '%', 'a', 'mais', 'por', 'o', 'direito', 'de', 'explorar', 'o', 'aeroporto', 'de', 'o', 'Galeão', '.'], ['Em', 'o', 'começo', 'de', 'o', 'século', ',', 'a', 'JBS/Friboi', 'chegava', 'a', 'o', 'grupo', 'de', 'as', '400', 'maiores', '.'], ['Os', 'sons', 'indesejáveis', 'emitidos', 'por', 'uma', 'porta', ',', 'por', 'exemplo', ',', 'são', 'eliminados', 'por', 'R$', '150', '.'], ['Que', 'foi', 'herança', 'de', 'o', 'PT', ',', 'que', 'nos', 'deixou', 'esse', 'rombo', ',', 'disse', 'Doria', '.'], ...])

In [ ]:
model

MultiTaskSentencePredictionEncoder(
  (bert): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (

In [ ]:
predict_test_df = get_predictions_on_dataframe(test_sentences, model, TOKENIZER)

100%|██████████| 1683/1683 [01:11<00:00, 23.65it/s]


In [ ]:
predict_test_df

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[O, Capitão, América, também, bajulou, o, tuca...","[det, nsubj, flat:name, advmod, root, det, obj...","[1.0, 0.9997614026069641, 0.9999967813491821, ...","[DET, PROPN, PROPN, ADV, VERB, DET, NOUN, PUNCT]","[1.0, 0.9999889135360718, 0.9999994039535522, ...","[2, 5, 2, 5, 0, 7, 5, 5]","[[tensor(2.3199e-08), tensor(9.0126e-09), tens..."
1,"[A, Odebrecht, pagou, 300, %, a, mais, por, o,...","[det, nsubj, root, nummod, obj, case, advmod, ...","[1.0, 0.9999991655349731, 0.9999998807907104, ...","[DET, PROPN, VERB, NUM, SYM, ADP, ADV, ADP, DE...","[1.0, 0.9999955892562866, 0.9999996423721313, ...","[2, 3, 0, 5, 3, 7, 5, 10, 10, 3, 12, 10, 14, 1...","[[tensor(6.4053e-09), tensor(7.5718e-09), tens..."
2,"[Em, o, começo, de, o, século, ,, a, JBS/Fribo...","[case, det, obl, case, det, nmod, punct, det, ...","[0.9999982118606567, 0.9999948740005493, 0.999...","[ADP, DET, NOUN, ADP, DET, NOUN, PUNCT, DET, P...","[0.9999988079071045, 0.9999967813491821, 0.999...","[3, 3, 12, 6, 6, 3, 3, 9, 12, 0, 15, 15, 12, 1...","[[tensor(2.0771e-08), tensor(3.8389e-09), tens..."
3,"[Os, sons, indesejáveis, emitidos, por, uma, p...","[det, nsubj:pass, amod, acl, case, det, obl:ag...","[0.9999998807907104, 0.9999468326568604, 0.999...","[DET, NOUN, ADJ, VERB, ADP, DET, NOUN, PUNCT, ...","[1.0, 0.9998999834060669, 0.9999996423721313, ...","[2, 13, 2, 2, 7, 7, 4, 10, 10, 13, 10, 13, 0, ...","[[tensor(6.8382e-08), tensor(5.3432e-08), tens..."
4,"[Que, foi, herança, de, o, PT, ,, que, nos, de...","[nsubj, cop, ccomp, case, det, nmod, punct, ns...","[0.9999761581420898, 0.9998980760574341, 0.991...","[PRON, AUX, NOUN, ADP, DET, PROPN, PUNCT, PRON...","[0.999988317489624, 0.9999945163726807, 0.9999...","[3, 3, 14, 6, 6, 3, 10, 10, 10, 6, 12, 10, 10,...","[[tensor(2.1570e-07), tensor(4.6328e-07), tens..."
...,...,...,...,...,...,...,...
1678,"[Julia, Louis-Dreyfus, ganhou, por, a, sexta, ...","[nsubj, flat:name, root, case, det, amod, obl,...","[0.9999773502349854, 0.9999998807907104, 0.999...","[PROPN, PROPN, VERB, ADP, DET, ADJ, NOUN, ADJ,...","[1.0, 1.0, 0.9999890327453613, 0.9999997615814...","[3, 1, 0, 7, 7, 7, 4, 7, 10, 3, 13, 13, 4, 16,...","[[tensor(1.4264e-05), tensor(2.4157e-07), tens..."
1679,"[Até, a, família, julga, mais, e, apoia, menos...","[advmod, det, nsubj, ccomp:speech, advmod, cc,...","[0.9984598159790039, 0.9999998807907104, 0.999...","[ADV, DET, NOUN, VERB, ADV, CCONJ, VERB, ADV, ...","[0.9961284399032593, 1.0, 0.9999970197677612, ...","[3, 3, 4, 15, 4, 7, 4, 7, 13, 11, 13, 13, 7, 4...","[[tensor(3.6238e-08), tensor(3.7275e-08), tens..."
1680,"[Mas, há, episódios, que, indicam, em, Damião,...","[cc, root, obj, nsubj, acl:relcl, case, obl, d...","[0.9999977350234985, 1.0, 0.9999994039535522, ...","[CCONJ, VERB, NOUN, PRON, VERB, ADP, PROPN, DE...","[0.9999960660934448, 0.9999992847442627, 0.999...","[2, 0, 2, 5, 3, 7, 5, 9, 5, 9, 2]","[[tensor(5.7678e-08), tensor(1.1305e-07), tens..."
1681,"["", Mas, Che, permanece, puro, ,, de, certo, m...","[punct, cc, nsubj, ccomp:speech, xcomp, punct,...","[0.999985933303833, 0.999840259552002, 0.99915...","[PUNCT, CCONJ, PROPN, VERB, ADJ, PUNCT, ADP, D...","[0.9999831914901733, 0.9998781681060791, 0.999...","[4, 4, 4, 13, 4, 9, 9, 9, 4, 4, 4, 13, 0, 13]","[[tensor(2.7810e-09), tensor(9.8504e-10), tens..."


In [ ]:
predict_test_df

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[O, Capitão, América, também, bajulou, o, tuca...","[det, nsubj, flat:name, advmod, root, det, obj...","[1.0, 0.9997614026069641, 0.9999967813491821, ...","[DET, PROPN, PROPN, ADV, VERB, DET, NOUN, PUNCT]","[1.0, 0.9999889135360718, 0.9999994039535522, ...","[2, 5, 2, 5, 0, 7, 5, 5]","[[tensor(2.3199e-08), tensor(9.0126e-09), tens..."
1,"[A, Odebrecht, pagou, 300, %, a, mais, por, o,...","[det, nsubj, root, nummod, obj, case, advmod, ...","[1.0, 0.9999991655349731, 0.9999998807907104, ...","[DET, PROPN, VERB, NUM, SYM, ADP, ADV, ADP, DE...","[1.0, 0.9999955892562866, 0.9999996423721313, ...","[2, 3, 0, 5, 3, 7, 5, 10, 10, 3, 12, 10, 14, 1...","[[tensor(6.4053e-09), tensor(7.5718e-09), tens..."
2,"[Em, o, começo, de, o, século, ,, a, JBS/Fribo...","[case, det, obl, case, det, nmod, punct, det, ...","[0.9999982118606567, 0.9999948740005493, 0.999...","[ADP, DET, NOUN, ADP, DET, NOUN, PUNCT, DET, P...","[0.9999988079071045, 0.9999967813491821, 0.999...","[3, 3, 12, 6, 6, 3, 3, 9, 12, 0, 15, 15, 12, 1...","[[tensor(2.0771e-08), tensor(3.8389e-09), tens..."
3,"[Os, sons, indesejáveis, emitidos, por, uma, p...","[det, nsubj:pass, amod, acl, case, det, obl:ag...","[0.9999998807907104, 0.9999468326568604, 0.999...","[DET, NOUN, ADJ, VERB, ADP, DET, NOUN, PUNCT, ...","[1.0, 0.9998999834060669, 0.9999996423721313, ...","[2, 13, 2, 2, 7, 7, 4, 10, 10, 13, 10, 13, 0, ...","[[tensor(6.8382e-08), tensor(5.3432e-08), tens..."
4,"[Que, foi, herança, de, o, PT, ,, que, nos, de...","[nsubj, cop, ccomp, case, det, nmod, punct, ns...","[0.9999761581420898, 0.9998980760574341, 0.991...","[PRON, AUX, NOUN, ADP, DET, PROPN, PUNCT, PRON...","[0.999988317489624, 0.9999945163726807, 0.9999...","[3, 3, 14, 6, 6, 3, 10, 10, 10, 6, 12, 10, 10,...","[[tensor(2.1570e-07), tensor(4.6328e-07), tens..."
...,...,...,...,...,...,...,...
1678,"[Julia, Louis-Dreyfus, ganhou, por, a, sexta, ...","[nsubj, flat:name, root, case, det, amod, obl,...","[0.9999773502349854, 0.9999998807907104, 0.999...","[PROPN, PROPN, VERB, ADP, DET, ADJ, NOUN, ADJ,...","[1.0, 1.0, 0.9999890327453613, 0.9999997615814...","[3, 1, 0, 7, 7, 7, 4, 7, 10, 3, 13, 13, 4, 16,...","[[tensor(1.4264e-05), tensor(2.4157e-07), tens..."
1679,"[Até, a, família, julga, mais, e, apoia, menos...","[advmod, det, nsubj, ccomp:speech, advmod, cc,...","[0.9984598159790039, 0.9999998807907104, 0.999...","[ADV, DET, NOUN, VERB, ADV, CCONJ, VERB, ADV, ...","[0.9961284399032593, 1.0, 0.9999970197677612, ...","[3, 3, 4, 15, 4, 7, 4, 7, 13, 11, 13, 13, 7, 4...","[[tensor(3.6238e-08), tensor(3.7275e-08), tens..."
1680,"[Mas, há, episódios, que, indicam, em, Damião,...","[cc, root, obj, nsubj, acl:relcl, case, obl, d...","[0.9999977350234985, 1.0, 0.9999994039535522, ...","[CCONJ, VERB, NOUN, PRON, VERB, ADP, PROPN, DE...","[0.9999960660934448, 0.9999992847442627, 0.999...","[2, 0, 2, 5, 3, 7, 5, 9, 5, 9, 2]","[[tensor(5.7678e-08), tensor(1.1305e-07), tens..."
1681,"["", Mas, Che, permanece, puro, ,, de, certo, m...","[punct, cc, nsubj, ccomp:speech, xcomp, punct,...","[0.999985933303833, 0.999840259552002, 0.99915...","[PUNCT, CCONJ, PROPN, VERB, ADJ, PUNCT, ADP, D...","[0.9999831914901733, 0.9998781681060791, 0.999...","[4, 4, 4, 13, 4, 9, 9, 9, 4, 4, 4, 13, 0, 13]","[[tensor(2.7810e-09), tensor(9.8504e-10), tens..."


In [ ]:
for i in range(1):
    print(
        f"Exemplo {i} - Tokens: {len(predict_test_df['tokens'][i])} | "
        #f"XPOS: {len(predict_test_df['xpos_predictions'][i])} | "
        f"DEPREL: {len(predict_test_df['deprel_predictions'][i])}"
    )


Exemplo 0 - Tokens: 8 | DEPREL: 8


In [ ]:
print(test_sentences[0])

['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [ ]:
def compute_dependency_metrics(test_sentences, test_upos, test_deprel, test_head, predict_df):
    """
    Métricas padrão para análise de dependências (dissertação):
      UPOS Accuracy — % de tokens com POS tag correta
      UAS           — Unlabeled Attachment Score: HEAD correto
      LAS           — Labeled Attachment Score:   HEAD + DEPREL corretos
    """
    total        = 0
    upos_correct = 0
    uas_correct  = 0
    las_correct  = 0
    skipped      = 0

    for i in range(len(test_sentences)):
        gold_upos   = test_upos[i]
        gold_deprel = test_deprel[i]
        gold_head   = test_head[i]

        pred_upos   = predict_df['upos_predictions'].iloc[i]
        pred_deprel = predict_df['deprel_predictions'].iloc[i]
        pred_head   = predict_df['head_predictions'].iloc[i]

        for j in range(len(gold_head)):
            if pred_head[j] is None or pred_upos[j] is None or pred_deprel[j] is None:
                skipped += 1
                continue

            total += 1

            if pred_upos[j] == gold_upos[j]:
                upos_correct += 1

            # UAS: HEAD correto
            if pred_head[j] == gold_head[j]:
                uas_correct += 1
                # LAS: HEAD correto E DEPREL correto
                if pred_deprel[j] == gold_deprel[j]:
                    las_correct += 1

    return {
        'upos_accuracy': upos_correct / total if total > 0 else 0,
        'uas':           uas_correct  / total if total > 0 else 0,
        'las':           las_correct  / total if total > 0 else 0,
        'total_tokens':  total,
        'skipped':       skipped,
    }


In [ ]:
metrics = compute_dependency_metrics(
    test_sentences, test_upos, test_deprel, test_head, predict_test_df
)

print(f"UPOS Accuracy : {metrics['upos_accuracy']:.4f}")
print(f"UAS           : {metrics['uas']:.4f}")
print(f"LAS           : {metrics['las']:.4f}")
print(f"Total tokens  : {metrics['total_tokens']}")
print(f"Ignorados     : {metrics['skipped']}")


UPOS Accuracy : 0.9881
UAS           : 0.8778
LAS           : 0.8625
Total tokens  : 33580
Ignorados     : 0


In [ ]:
#text = [["O", "gato", "preto", "dorme", "no", "sofá", "."]]
#text = [["A", "menina", "brinca", "no", "parque", "."]]
#text = [["Se", "chover", ",", "o", "jogo", "será", "cancelado", "."]]
text = [['Mas', 'por', 'não', 'existir', 'um', 'marco', 'legal', 'há', 'uma', 'insegurança', 'por', 'parte', 'dos', 'investidores', '"', ',', 'destacou', '.']]

In [ ]:
#text = [[".", ".", ".", ".", "", ""]]

#Token = Token_alvo Indice
#Optuna

In [ ]:
retorno = get_predictions_on_dataframe(text, model, TOKENIZER)
#aux = [test_df['tokens'][0]]


100%|██████████| 1/1 [00:00<00:00, 20.89it/s]


In [ ]:
#retorno = get_predictions_on_dataframe(aux, model, TOKENIZER)

In [ ]:
retorno

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[Mas, por, não, existir, um, marco, legal, há,...","[cc, mark, advmod, advcl, det, nsubj, amod, cc...","[0.9999947547912598, 0.9999908208847046, 0.999...","[CCONJ, ADP, ADV, VERB, DET, NOUN, ADJ, VERB, ...","[0.9999758005142212, 0.9999992847442627, 0.999...","[8, 4, 4, 8, 6, 4, 6, 17, 10, 8, 12, 8, 14, 12...","[[tensor(6.1319e-07), tensor(8.1826e-06), tens..."


In [ ]:
retorno

,tokens,deprel_predictions,deprel_pred_probability,upos_predictions,upos_pred_probability,head_predictions,head_pred_probability
0,"[Mas, por, não, existir, um, marco, legal, há,...","[cc, mark, advmod, advcl, det, nsubj, amod, cc...","[0.9999947547912598, 0.9999908208847046, 0.999...","[CCONJ, ADP, ADV, VERB, DET, NOUN, ADJ, VERB, ...","[0.9999758005142212, 0.9999992847442627, 0.999...","[8, 4, 4, 8, 6, 4, 6, 17, 10, 8, 12, 8, 14, 12...","[[tensor(6.1319e-07), tensor(8.1826e-06), tens..."


In [ ]:
print('Gold deprel sentença 2:', test_deprel[2])
print('Pred deprel sentença 2:', predict_test_df['deprel_predictions'].iloc[2])

Gold deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']
Pred deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']


In [ ]:
predict_test_df.to_csv('./predict_test_df_jabuticabert_linear.csv', index=False)

# Análise por camada (Logit Lens)

**Objetivo:** avaliar o que **cada camada do BERT** produz em relação à label final que queremos inferir.
Exemplo: entra o token `'O'` (gold = `DET`) — como cada uma das camadas classifica esse token?
E **qual camada empurrou a predição na direção correta**?

**Método (logit lens):** rodamos o encoder com `output_hidden_states=True` e aplicamos os
**mesmos classificadores lineares treinados** (`upos_classifier`, `deprel_classifier`, `head_classifier`)
sobre a saída de cada camada:

- camada `0` = embeddings (antes do encoder)
- camadas `1..12` = saída de cada bloco Transformer

> ⚠️ Nota metodológica: os classificadores foram treinados apenas sobre a **última camada**,
> então as camadas intermediárias são lidas "pela lente" do classificador final (logit lens clássico,
> Nostalgebraist 2020). A trajetória da probabilidade da label gold ao longo das camadas indica
> **onde** a informação necessária para a decisão se forma dentro do encoder.

Modelo usado nesta seção: **`linear_BERTimbau_base`**.

In [ ]:
from transformers import AutoConfig, AutoTokenizer

# ── Modelo para análise por camada: linear_BERTimbau_base ─────────────────────
LAYERWISE_MODEL_PATH     = '/home/guilhermelima/msc/linear_BERTimbau_base'
LAYERWISE_TOKENIZER_NAME = 'neuralmind/bert-base-portuguese-cased'  # BERTimbau-base

layerwise_config    = AutoConfig.from_pretrained(LAYERWISE_MODEL_PATH)
layerwise_tokenizer = AutoTokenizer.from_pretrained(LAYERWISE_TOKENIZER_NAME)

layerwise_model = MultiTaskSentencePredictionEncoder.from_pretrained(
    LAYERWISE_MODEL_PATH,
    config=layerwise_config,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS)
).to("cuda" if torch.cuda.is_available() else "cpu")
layerwise_model.eval()

NUM_HIDDEN_LAYERS = layerwise_config.num_hidden_layers  # 12 no BERTimbau-base
print(f'Arquitetura: {layerwise_config.model_type} | camadas: {NUM_HIDDEN_LAYERS} | hidden: {layerwise_config.hidden_size}')


In [ ]:
def get_layerwise_predictions(tokens, model, tokenizer,
                              gold_upos=None, gold_deprel=None, gold_head=None,
                              device=None):
    """
    Aplica os classificadores treinados sobre a saída de CADA camada do BERT.

    Retorna um DataFrame longo com uma linha por (token, camada, tarefa):
      layer      : 0 = embeddings, 1..12 = camadas do encoder
      pred       : label predita pela camada
      pred_prob  : probabilidade (softmax) da label predita
      gold       : label correta (se fornecida)
      gold_prob  : probabilidade que a camada atribui à label correta
      correct    : predição da camada == gold
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    inputs = tokenizer(
        tokens, is_split_into_words=True, return_tensors="pt",
        padding=True, truncation=True
    ).to(device)

    with torch.no_grad():
        bert_out = model.bert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            token_type_ids=inputs.get('token_type_ids'),
            output_hidden_states=True,
        )
        hidden_states = bert_out.hidden_states  # tupla de 13 tensores [1, seq_len, hidden]

        # logits de cada camada com os MESMOS classificadores finais (logit lens)
        layer_logits = [
            {
                'upos':   model.upos_classifier(hs),
                'deprel': model.deprel_classifier(hs),
                'head':   model.head_classifier(hs),
            }
            for hs in hidden_states
        ]

    word_ids = inputs.word_ids(batch_index=0)

    tasks = [
        ('upos',   IDX_TO_UPOS_LABELS,   UPOS_LABELS_TO_IDX,   gold_upos),
        ('deprel', IDX_TO_DEPREL_LABELS, DEPREL_LABELS_TO_IDX, gold_deprel),
        ('head',   None,                 None,                 gold_head),
    ]

    rows = []
    for token_idx, token in enumerate(tokens):
        subtoken_idxs = [i for i, w in enumerate(word_ids) if w == token_idx]
        if not subtoken_idxs:
            continue
        first_sub = subtoken_idxs[0]  # mesma estratégia da inferência normal

        for layer_idx, logits in enumerate(layer_logits):
            for task, id2label, label2id, gold_seq in tasks:
                probs = torch.softmax(logits[task][0, first_sub], dim=-1)
                pred_id = torch.argmax(probs).item()
                pred = id2label[pred_id] if id2label is not None else pred_id

                gold, gold_prob, correct = None, None, None
                if gold_seq is not None:
                    gold = gold_seq[token_idx]
                    gold_id = label2id[gold] if label2id is not None else int(gold)
                    gold_prob = probs[gold_id].item()
                    correct = (pred_id == gold_id)

                rows.append({
                    'token_idx': token_idx,
                    'token': token,
                    'layer': layer_idx,
                    'task': task,
                    'pred': pred,
                    'pred_prob': probs[pred_id].item(),
                    'gold': gold,
                    'gold_prob': gold_prob,
                    'correct': correct,
                })

    return pd.DataFrame(rows)


In [ ]:
def inspect_token_across_layers(df_layers, token_idx, task='upos'):
    """
    Mostra, camada a camada, como o modelo classifica UM token para UMA tarefa.
    Responde: "entra 'O' (gold DET) — o que cada camada infere sobre ele?"
    """
    sel = (df_layers[(df_layers.token_idx == token_idx) & (df_layers.task == task)]
           .sort_values('layer').reset_index(drop=True))

    token = sel['token'].iloc[0]
    gold  = sel['gold'].iloc[0]
    print(f"Token: '{token}' (idx {token_idx}) | tarefa: {task} | gold: {gold}")

    tabela = sel[['layer', 'pred', 'pred_prob', 'gold_prob', 'correct']].copy()
    tabela['pred_prob'] = tabela['pred_prob'].round(4)
    if gold is not None:
        tabela['gold_prob'] = tabela['gold_prob'].round(4)
    return tabela


def layer_influence(df_layers, token_idx, task='upos'):
    """
    Mede a INFLUÊNCIA de cada camada na direção da label correta:
      delta_gold_prob[i] = P_gold(camada i) - P_gold(camada i-1)

    - delta > 0  → a camada empurrou a predição NA DIREÇÃO CORRETA
    - delta < 0  → a camada afastou a predição da label correta

    Também reporta:
      - camada de maior contribuição positiva
      - 'camada de decisão': primeira camada a partir da qual a predição
        fica correta e assim permanece até a última camada
    """
    sel = (df_layers[(df_layers.token_idx == token_idx) & (df_layers.task == task)]
           .sort_values('layer').reset_index(drop=True))

    token = sel['token'].iloc[0]
    gold  = sel['gold'].iloc[0]
    gp    = sel['gold_prob'].to_numpy(dtype=float)
    corr  = sel['correct'].to_numpy(dtype=bool)

    deltas = np.diff(gp)  # contribuição das camadas 1..N

    # camada de decisão: última transição incorreto->correto
    decision_layer = None
    for l in range(len(corr)):
        if corr[l:].all():
            decision_layer = l
            break

    best_layer = int(np.argmax(deltas)) + 1

    print(f"Token: '{token}' | tarefa: {task} | gold: {gold}")
    print(f"P(gold) embeddings (camada 0): {gp[0]:.4f}")
    print(f"P(gold) camada final          : {gp[-1]:.4f}")
    print(f"Camada de MAIOR contribuição positiva: {best_layer} (Δ = +{deltas[best_layer-1]:.4f})")
    if decision_layer is not None:
        print(f"Camada de decisão (predição correta e estável a partir dela): {decision_layer}")
    else:
        print("Predição final INCORRETA — nenhuma camada estabilizou na label gold.")

    df_infl = pd.DataFrame({
        'layer': np.arange(1, len(gp)),
        'delta_gold_prob': np.round(deltas, 4),
        'gold_prob_acumulada': np.round(gp[1:], 4),
        'pred_da_camada': sel['pred'].iloc[1:].values,
    })
    return df_infl


In [ ]:
# ── Exemplo: token 'O' com gold 'DET' ──────────────────────────────────────────
# Procura no test set a primeira sentença com token 'O' anotado como DET
exemplo_sent_idx, exemplo_tok_idx = None, None
for i, (toks, upos_seq) in enumerate(zip(test_sentences, test_upos)):
    for j, (t, u) in enumerate(zip(toks, upos_seq)):
        if t == 'O' and u == 'DET':
            exemplo_sent_idx, exemplo_tok_idx = i, j
            break
    if exemplo_sent_idx is not None:
        break

print(f"Sentença {exemplo_sent_idx}: {test_sentences[exemplo_sent_idx]}")
print(f"Token alvo: '{test_sentences[exemplo_sent_idx][exemplo_tok_idx]}' (posição {exemplo_tok_idx}) | gold UPOS: {test_upos[exemplo_sent_idx][exemplo_tok_idx]}")

df_layers_exemplo = get_layerwise_predictions(
    test_sentences[exemplo_sent_idx],
    layerwise_model,
    layerwise_tokenizer,
    gold_upos=test_upos[exemplo_sent_idx],
    gold_deprel=test_deprel[exemplo_sent_idx],
    gold_head=test_head[exemplo_sent_idx],
)
df_layers_exemplo.head()


In [ ]:
# Como cada camada classifica o 'O' (gold DET) para UPOS?
inspect_token_across_layers(df_layers_exemplo, exemplo_tok_idx, task='upos')


In [ ]:
# Qual camada influenciou na direção correta?
layer_influence(df_layers_exemplo, exemplo_tok_idx, task='upos')


In [ ]:
import matplotlib.pyplot as plt

def plot_token_trajectory(df_layers, token_idx, tasks=('upos', 'deprel', 'head')):
    """Trajetória de P(label gold) ao longo das camadas, por tarefa, para um token."""
    fig, ax = plt.subplots(figsize=(9, 5))
    token = df_layers[df_layers.token_idx == token_idx]['token'].iloc[0]

    for task in tasks:
        sel = (df_layers[(df_layers.token_idx == token_idx) & (df_layers.task == task)]
               .sort_values('layer'))
        gold = sel['gold'].iloc[0]
        ax.plot(sel['layer'], sel['gold_prob'], marker='o', label=f"{task} (gold: {gold})")

        # marca camadas onde a predição está correta
        ok = sel[sel['correct'] == True]
        ax.scatter(ok['layer'], ok['gold_prob'], s=110, facecolors='none',
                   edgecolors='green', linewidths=1.8, zorder=3)

    ax.set_xlabel('Camada (0 = embeddings)')
    ax.set_ylabel('P(label gold)')
    ax.set_title(f"Trajetória da probabilidade da label gold — token '{token}'\n(círculo verde = camada acerta a predição)")
    ax.set_xticks(range(df_layers['layer'].max() + 1))
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_token_trajectory(df_layers_exemplo, exemplo_tok_idx)


In [ ]:
def plot_sentence_layer_heatmap(df_layers, task='upos'):
    """
    Heatmap tokens x camadas com P(label gold).
    Cada célula é anotada com a label PREDITA pela camada — permite ver, token a
    token, em que profundidade a predição converge para a label correta.
    """
    sel = df_layers[df_layers.task == task]
    piv_prob = sel.pivot(index='token_idx', columns='layer', values='gold_prob')
    piv_pred = sel.pivot(index='token_idx', columns='layer', values='pred')
    piv_corr = sel.pivot(index='token_idx', columns='layer', values='correct')

    tokens = sel.drop_duplicates('token_idx').set_index('token_idx')
    ylabels = [f"{tokens.loc[i, 'token']} ({tokens.loc[i, 'gold']})" for i in piv_prob.index]

    n_tok, n_lay = piv_prob.shape
    fig, ax = plt.subplots(figsize=(1.15 * n_lay, 0.5 * n_tok + 1.5))
    im = ax.imshow(piv_prob.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

    ax.set_xticks(range(n_lay), labels=piv_prob.columns)
    ax.set_yticks(range(n_tok), labels=ylabels)
    ax.set_xlabel('Camada (0 = embeddings)')
    ax.set_title(f"P(gold) e predição de cada camada — tarefa: {task}\n(negrito = camada acerta a label)")

    for r in range(n_tok):
        for c in range(n_lay):
            pred = str(piv_pred.values[r, c])
            ok = bool(piv_corr.values[r, c])
            ax.text(c, r, pred, ha='center', va='center',
                    fontsize=7, fontweight='bold' if ok else 'normal',
                    color='black' if ok else 'dimgray')

    fig.colorbar(im, ax=ax, label='P(label gold)')
    plt.tight_layout()
    plt.show()

plot_sentence_layer_heatmap(df_layers_exemplo, task='upos')


In [ ]:
# Mesma visualização para DEPREL e HEAD
plot_sentence_layer_heatmap(df_layers_exemplo, task='deprel')
plot_sentence_layer_heatmap(df_layers_exemplo, task='head')


## Análise agregada: acurácia por camada no test set

A análise acima é por token/sentença. Para uma visão global de **qual camada resolve cada tarefa**,
calculamos a acurácia de cada camada (via logit lens) sobre uma amostra do test set:

- camadas onde a acurácia **salta** são as que mais adicionam informação para a tarefa;
- tarefas mais "sintáticas" (head/deprel) tendem a se resolver em camadas mais profundas que UPOS.

In [ ]:
def layerwise_accuracy_on_dataset(sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                                  model, tokenizer, max_sentences=300, device=None):
    """
    Acurácia por camada (logit lens) para upos/deprel/head sobre uma amostra do dataset.
    Retorna DataFrame: layer x tarefa, além da média de P(gold) por camada.
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    n_layers = model.config.num_hidden_layers + 1  # + embeddings
    correct = {t: np.zeros(n_layers) for t in ('upos', 'deprel', 'head')}
    gold_prob_sum = {t: np.zeros(n_layers) for t in ('upos', 'deprel', 'head')}
    total = 0

    n = min(max_sentences, len(sentences))
    for i in tqdm(range(n)):
        tokens = sentences[i]
        golds = {
            'upos':   [UPOS_LABELS_TO_IDX[u] for u in gold_upos_list[i]],
            'deprel': [DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]],
            'head':   [int(h) for h in gold_head_list[i]],
        }

        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            bert_out = model.bert(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids'),
                output_hidden_states=True,
            )
            hidden_states = bert_out.hidden_states

            word_ids = inputs.word_ids(batch_index=0)
            # primeiro subtoken de cada token (mesma estratégia da inferência normal)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)

            if not first_subs:
                continue
            first_subs = torch.tensor(first_subs, device=device)

            for layer_idx, hs in enumerate(hidden_states):
                hs_tok = hs[0, first_subs]  # [n_tokens, hidden]
                for task, classifier in (('upos', model.upos_classifier),
                                         ('deprel', model.deprel_classifier),
                                         ('head', model.head_classifier)):
                    probs = torch.softmax(classifier(hs_tok), dim=-1)
                    preds = probs.argmax(dim=-1)
                    gold_ids = torch.tensor([golds[task][t] for t in tok_idxs], device=device)

                    correct[task][layer_idx] += (preds == gold_ids).sum().item()
                    gold_prob_sum[task][layer_idx] += probs.gather(1, gold_ids.unsqueeze(1)).sum().item()

        total += len(first_subs)

    acc = pd.DataFrame({f'{t}_acc': correct[t] / total for t in correct})
    acc.insert(0, 'layer', np.arange(n_layers))
    for t in gold_prob_sum:
        acc[f'{t}_mean_gold_prob'] = gold_prob_sum[t] / total
    return acc


layer_acc_df = layerwise_accuracy_on_dataset(
    test_sentences, test_upos, test_deprel, test_head,
    layerwise_model, layerwise_tokenizer,
    max_sentences=300,   # aumente para o test set completo se quiser
)
layer_acc_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for task, color in (('upos', 'tab:blue'), ('deprel', 'tab:orange'), ('head', 'tab:green')):
    axes[0].plot(layer_acc_df['layer'], layer_acc_df[f'{task}_acc'],
                 marker='o', color=color, label=task)
    axes[1].plot(layer_acc_df['layer'][1:], np.diff(layer_acc_df[f'{task}_acc']),
                 marker='s', color=color, label=task)

axes[0].set_title('Acurácia por camada (logit lens)')
axes[0].set_xlabel('Camada (0 = embeddings)')
axes[0].set_ylabel('Acurácia')
axes[1].set_title('Ganho de acurácia por camada (Δ vs. camada anterior)\n= influência de cada camada na direção correta')
axes[1].set_xlabel('Camada')
axes[1].set_ylabel('Δ acurácia')
axes[1].axhline(0, color='gray', lw=0.8)

for ax in axes:
    ax.set_xticks(range(len(layer_acc_df)))
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

# Camada de maior ganho por tarefa
for task in ('upos', 'deprel', 'head'):
    d = np.diff(layer_acc_df[f'{task}_acc'])
    print(f"{task:7s} → camada de maior ganho: {np.argmax(d) + 1} (Δ = +{d.max():.4f}) | acc final: {layer_acc_df[f'{task}_acc'].iloc[-1]:.4f}")


## Impacto por camada em TODO o test set — por tag e por tarefa

Extrapolação da análise anterior para o dataset completo. Para cada **tarefa** (upos, deprel, head)
e cada **tag** (label gold), computamos em cada camada:

- **acc** — acurácia da camada para tokens com aquela tag gold (logit lens);
- **mean_gold_prob** — probabilidade média que a camada atribui à tag correta.

Métricas de impacto por tag:

- **best_layer_acc** — camada com maior ganho de acurácia (Δacc vs. camada anterior);
- **best_layer_prob** — camada com maior ganho de P(gold) — *a camada que mais empurrou na direção correta*;
- **conv_layer** — camada de convergência: primeira camada com acc ≥ 95% da acurácia final.

> Para a tarefa `head`, as "tags" são as posições-alvo (0 = root, 1..N = índice do head na sentença).

In [ ]:
def layerwise_per_label_stats(sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                              model, tokenizer, device=None, max_sentences=None):
    """
    Estatísticas por (tarefa, tag gold, camada) sobre o dataset via logit lens.

    Retorna dict {task: DataFrame} em formato longo:
      label | layer | acc | mean_gold_prob | support
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    n_layers = model.config.num_hidden_layers + 1  # + embeddings
    task_info = {
        'upos':   (model.upos_classifier,   len(UPOS_LABELS)),
        'deprel': (model.deprel_classifier, len(DEPREL_LABELS)),
        'head':   (model.head_classifier,   model.num_head_labels),
    }
    correct       = {t: np.zeros((n_lab, n_layers)) for t, (_, n_lab) in task_info.items()}
    gold_prob_sum = {t: np.zeros((n_lab, n_layers)) for t, (_, n_lab) in task_info.items()}
    support       = {t: np.zeros(n_lab)             for t, (_, n_lab) in task_info.items()}

    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n)):
        tokens = sentences[i]
        golds = {
            'upos':   [UPOS_LABELS_TO_IDX[u]   for u in gold_upos_list[i]],
            'deprel': [DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]],
            'head':   [int(h)                  for h in gold_head_list[i]],
        }

        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            bert_out = model.bert(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids'),
                output_hidden_states=True,
            )
            # [n_layers, seq_len, hidden]
            hs_all = torch.stack(bert_out.hidden_states, dim=0)[:, 0]

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue

            hs_tok = hs_all[:, torch.tensor(first_subs, device=device)]  # [n_layers, n_tok, hidden]

            for task, (classifier, n_lab) in task_info.items():
                gold_ids_np = np.array([golds[task][t] for t in tok_idxs])
                gold_ids = torch.tensor(gold_ids_np, device=device)

                probs = torch.softmax(classifier(hs_tok), dim=-1)      # [n_layers, n_tok, n_lab]
                preds = probs.argmax(dim=-1)                           # [n_layers, n_tok]
                gp = probs.gather(-1, gold_ids.expand(len(probs), -1).unsqueeze(-1)).squeeze(-1)

                corr_np = (preds == gold_ids).cpu().numpy()            # [n_layers, n_tok]
                gp_np   = gp.cpu().numpy()

                for layer_idx in range(len(probs)):
                    np.add.at(correct[task][:, layer_idx],       gold_ids_np, corr_np[layer_idx])
                    np.add.at(gold_prob_sum[task][:, layer_idx], gold_ids_np, gp_np[layer_idx])
                np.add.at(support[task], gold_ids_np, 1)

    id2label = {'upos': IDX_TO_UPOS_LABELS, 'deprel': IDX_TO_DEPREL_LABELS, 'head': None}
    stats = {}
    for task, (_, n_lab) in task_info.items():
        rows = []
        for lab_id in range(n_lab):
            sup = support[task][lab_id]
            if sup == 0:
                continue
            label = id2label[task][lab_id] if id2label[task] is not None else lab_id
            for layer in range(n_layers):
                rows.append({
                    'label': label, 'layer': layer,
                    'acc':            correct[task][lab_id, layer] / sup,
                    'mean_gold_prob': gold_prob_sum[task][lab_id, layer] / sup,
                    'support': int(sup),
                })
        stats[task] = pd.DataFrame(rows)
    return stats


In [ ]:
# ⏱ Roda no test set COMPLETO (1683 sentenças, ~33,5k tokens)
perlabel_stats = layerwise_per_label_stats(
    test_sentences, test_upos, test_deprel, test_head,
    layerwise_model, layerwise_tokenizer,
)

for task, df in perlabel_stats.items():
    df.to_csv(f'./layerwise_per_label_stats_{task}.csv', index=False)
    print(f'{task}: {df.label.nunique()} tags | salvo em layerwise_per_label_stats_{task}.csv')


In [ ]:
def summarize_layer_impact(stats_df, min_support=30):
    """
    Para cada tag: qual camada foi mais impactante?
      best_layer_acc  — camada com maior Δ acurácia
      best_layer_prob — camada com maior Δ P(gold)  (mais empurrou na direção correta)
      conv_layer      — primeira camada com acc >= 95% da acurácia final
    """
    rows = []
    for label, g in stats_df.groupby('label', sort=False):
        g = g.sort_values('layer')
        sup = g['support'].iloc[0]
        if sup < min_support:
            continue
        acc = g['acc'].to_numpy()
        gp  = g['mean_gold_prob'].to_numpy()
        d_acc, d_gp = np.diff(acc), np.diff(gp)

        final_acc = acc[-1]
        conv = next((l for l in range(len(acc)) if final_acc > 0 and acc[l] >= 0.95 * final_acc), None)

        rows.append({
            'label': label, 'support': sup,
            'acc_emb': round(acc[0], 4), 'acc_final': round(final_acc, 4),
            'best_layer_acc':  int(np.argmax(d_acc)) + 1, 'delta_acc_max':  round(d_acc.max(), 4),
            'best_layer_prob': int(np.argmax(d_gp))  + 1, 'delta_prob_max': round(d_gp.max(), 4),
            'conv_layer': conv,
        })
    return pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)


impact_summary = {}
for task, min_sup in (('upos', 30), ('deprel', 30), ('head', 100)):
    impact_summary[task] = summarize_layer_impact(perlabel_stats[task], min_support=min_sup)
    impact_summary[task].to_csv(f'./layerwise_impact_summary_{task}.csv', index=False)

print('═' * 80)
print('UPOS — camadas mais impactantes por tag')
display(impact_summary['upos'])
print('═' * 80)
print('DEPREL — camadas mais impactantes por tag (support >= 30)')
display(impact_summary['deprel'])
print('═' * 80)
print('HEAD — camadas mais impactantes por posição-alvo (support >= 100)')
display(impact_summary['head'])


In [ ]:
import matplotlib.pyplot as plt

def plot_perlabel_heatmap(stats_df, task, metric='acc', top_n=20, min_support=30):
    """Heatmap tags x camadas (tags ordenadas por frequência)."""
    sup = stats_df.drop_duplicates('label').set_index('label')['support']
    labels = sup[sup >= min_support].sort_values(ascending=False).head(top_n).index

    piv = (stats_df[stats_df.label.isin(labels)]
           .pivot(index='label', columns='layer', values=metric)
           .loc[labels])

    fig, ax = plt.subplots(figsize=(11, 0.42 * len(piv) + 1.8))
    im = ax.imshow(piv.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(piv.shape[1]), labels=piv.columns)
    ax.set_yticks(range(len(piv)), labels=[f'{l} (n={sup[l]})' for l in piv.index])
    ax.set_xlabel('Camada (0 = embeddings)')
    ax.set_title(f'{task.upper()} — {metric} por tag x camada (test set completo)')
    fig.colorbar(im, ax=ax, label=metric)
    plt.tight_layout()
    plt.show()

for task in ('upos', 'deprel', 'head'):
    plot_perlabel_heatmap(perlabel_stats[task], task, metric='acc',
                          min_support=30 if task != 'head' else 100)


In [ ]:
# ── Síntese: onde cada tarefa é "resolvida" dentro do BERT ─────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
width = 0.27
for k, (task, color) in enumerate((('upos', 'tab:blue'), ('deprel', 'tab:orange'), ('head', 'tab:green'))):
    counts = impact_summary[task]['best_layer_prob'].value_counts().reindex(range(1, 13), fill_value=0)
    ax.bar(np.arange(1, 13) + (k - 1) * width, counts.values, width, color=color, label=task)

ax.set_xlabel('Camada mais impactante (maior Δ P(gold))')
ax.set_ylabel('Nº de tags')
ax.set_title('Distribuição da camada mais impactante por tarefa')
ax.set_xticks(range(1, 13))
ax.grid(alpha=0.3, axis='y')
ax.legend()
plt.tight_layout()
plt.show()

print('Média (ponderada por support) da camada mais impactante:')
for task in ('upos', 'deprel', 'head'):
    s = impact_summary[task]
    w_prob = np.average(s['best_layer_prob'], weights=s['support'])
    w_conv = np.average(s['conv_layer'].astype(float), weights=s['support'])
    print(f"  {task:7s} → impacto: camada {w_prob:.1f} | convergência: camada {w_conv:.1f}")


## Usar apenas as melhores camadas é suficiente? (Early Exit vs. profundidade total)

Pergunta: **é melhor usar diretamente a camada que converge melhor para cada tarefa,
ou é necessário passar por todas as camadas?**

Estratégia: coletamos as predições (argmax) de **todas as 13 camadas** para as 3 tarefas em
uma única passada pelo test set. Depois avaliamos offline qualquer combinação de camadas:

1. **Baseline** — todas as tarefas na camada final (12) = inferência normal;
2. **Melhor camada por tarefa** — cada tarefa lê da sua camada de maior acurácia global;
3. **Camadas de convergência** — cada tarefa lê da camada de convergência (análise anterior);
4. **Early exit compartilhado** — truncar o encoder na camada L e ler as 3 tarefas dali
   (curva L = 0..12, mede o custo real de "não passar por todas as camadas").

Métricas: acurácia por tarefa + **UAS/LAS** (as métricas oficiais da dissertação).

> ⚠️ Os classificadores foram treinados sobre a camada 12 — os resultados das camadas
> intermediárias são um limite inferior (com fine-tuning/probes por camada poderiam ser maiores).

In [ ]:
def collect_predictions_all_layers(sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                                   model, tokenizer, device=None, max_sentences=None):
    """
    Uma única passada: guarda a predição (argmax) de CADA camada para CADA tarefa,
    por token. Permite avaliar offline qualquer combinação de camadas.

    Retorna (preds_store, golds_store):
      preds_store[i][task] -> np.array [n_layers, n_tokens_validos]
      golds_store[i][task] -> np.array [n_tokens_validos]
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    classifiers = {
        'upos':   model.upos_classifier,
        'deprel': model.deprel_classifier,
        'head':   model.head_classifier,
    }
    preds_store, golds_store = [], []

    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n)):
        tokens = sentences[i]
        golds = {
            'upos':   [UPOS_LABELS_TO_IDX[u]   for u in gold_upos_list[i]],
            'deprel': [DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]],
            'head':   [int(h)                  for h in gold_head_list[i]],
        }

        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            bert_out = model.bert(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids'),
                output_hidden_states=True,
            )
            hs_all = torch.stack(bert_out.hidden_states, dim=0)[:, 0]  # [n_layers, seq, hidden]

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue

            hs_tok = hs_all[:, torch.tensor(first_subs, device=device)]  # [n_layers, n_tok, hidden]

            sent_preds, sent_golds = {}, {}
            for task, clf in classifiers.items():
                sent_preds[task] = clf(hs_tok).argmax(dim=-1).cpu().numpy().astype(np.int16)
                sent_golds[task] = np.array([golds[task][t] for t in tok_idxs], dtype=np.int16)

            preds_store.append(sent_preds)
            golds_store.append(sent_golds)

    return preds_store, golds_store


# Uma passada no test set completo (as 13 camadas ficam salvas em memória, ~poucos MB)
preds_store, golds_store = collect_predictions_all_layers(
    test_sentences, test_upos, test_deprel, test_head,
    layerwise_model, layerwise_tokenizer,
)
print(f'{len(preds_store)} sentenças coletadas | camadas: {preds_store[0]["upos"].shape[0]}')


In [ ]:
def eval_layer_combo(preds_store, golds_store, upos_layer, deprel_layer, head_layer):
    """
    Avalia a combinação de camadas (uma por tarefa) com as métricas da dissertação:
    acurácia por tarefa + UAS + LAS.
    """
    total = upos_c = deprel_c = uas_c = las_c = 0
    for preds, golds in zip(preds_store, golds_store):
        p_upos, p_deprel, p_head = preds['upos'][upos_layer], preds['deprel'][deprel_layer], preds['head'][head_layer]
        g_upos, g_deprel, g_head = golds['upos'], golds['deprel'], golds['head']

        total    += len(g_upos)
        upos_c   += (p_upos == g_upos).sum()
        deprel_c += (p_deprel == g_deprel).sum()
        head_ok   = (p_head == g_head)
        uas_c    += head_ok.sum()
        las_c    += (head_ok & (p_deprel == g_deprel)).sum()

    return {
        'upos_layer': upos_layer, 'deprel_layer': deprel_layer, 'head_layer': head_layer,
        'upos_acc':   upos_c / total,
        'deprel_acc': deprel_c / total,
        'uas':        uas_c / total,
        'las':        las_c / total,
    }


# ── Acurácia global de cada camada por tarefa (test set completo) ──────────────
n_layers_total = preds_store[0]['upos'].shape[0]
per_layer_acc = pd.DataFrame([
    eval_layer_combo(preds_store, golds_store, L, L, L) for L in range(n_layers_total)
]).rename(columns={'upos_layer': 'layer'}).drop(columns=['deprel_layer', 'head_layer'])

BEST_LAYER = {
    'upos':   int(per_layer_acc['upos_acc'].idxmax()),
    'deprel': int(per_layer_acc['deprel_acc'].idxmax()),
    'head':   int(per_layer_acc['uas'].idxmax()),
}
print('Melhor camada por tarefa (acurácia global):', BEST_LAYER)
per_layer_acc.round(4)


In [ ]:
# ── Comparação: melhores camadas diretamente vs. passar por todas ─────────────
FINAL = n_layers_total - 1  # camada 12

configs = {
    'baseline: tudo na camada final (12)': (FINAL, FINAL, FINAL),
    'melhor camada por tarefa':            (BEST_LAYER['upos'], BEST_LAYER['deprel'], BEST_LAYER['head']),
    'camadas de convergência (8, 9, 12)':  (8, 9, FINAL),
    'early exit na camada 10':             (10, 10, 10),
    'early exit na camada 9':              (9, 9, 9),
    'early exit na camada 8':              (8, 8, 8),
}

rows = []
for name, (lu, ld, lh) in configs.items():
    r = eval_layer_combo(preds_store, golds_store, lu, ld, lh)
    r['config'] = name
    rows.append(r)

comparison = pd.DataFrame(rows)[
    ['config', 'upos_layer', 'deprel_layer', 'head_layer', 'upos_acc', 'deprel_acc', 'uas', 'las']
]
comparison.to_csv('./layerwise_early_exit_comparison.csv', index=False)
comparison.round(4)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5.5))
for metric, color in (('upos_acc', 'tab:blue'), ('deprel_acc', 'tab:orange'),
                      ('uas', 'tab:green'), ('las', 'tab:red')):
    ax.plot(per_layer_acc['layer'], per_layer_acc[metric], marker='o', color=color, label=metric)
    best_l = int(per_layer_acc[metric].idxmax())
    ax.scatter([best_l], [per_layer_acc[metric].max()], s=160, facecolors='none',
               edgecolors=color, linewidths=2, zorder=3)

base = per_layer_acc.iloc[-1]
ax.set_xlabel('Early exit: encoder truncado na camada L (3 tarefas lidas de L)')
ax.set_ylabel('Métrica no test set completo')
ax.set_title('Custo de NÃO passar por todas as camadas\n(círculo = melhor camada para a métrica)')
ax.set_xticks(range(n_layers_total))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print('Perda ao usar early exit (vs. camada final):')
for L in (8, 9, 10, 11):
    r = per_layer_acc.iloc[L]
    print(f"  camada {L:2d} → UPOS {r.upos_acc - base.upos_acc:+.4f} | "
          f"DEPREL {r.deprel_acc - base.deprel_acc:+.4f} | "
          f"UAS {r.uas - base.uas:+.4f} | LAS {r.las - base.las:+.4f}")


## É possível ativar APENAS a camada 12? (Block skip)

Experimento complementar ao early exit: em vez de truncar o encoder no topo,
**pulamos blocos no meio** — as embeddings entram diretamente em blocos escolhidos.

- `embeddings → bloco 12 → classificadores` (só a camada 12 "ativa")
- `embeddings → blocos 11,12 → classificadores`
- etc.

Mecanicamente é possível (cada bloco é um módulo independente que mapeia `[seq, 768] → [seq, 768]`).
A questão é se o bloco 12 funciona sem receber a representação construída pelos blocos 1–11 —
os pesos dele foram treinados esperando exatamente essa entrada.

> Sanity check incluído: `embeddings → blocos 1..12` reproduz exatamente a inferência normal.

In [ ]:
def eval_block_subset(blocks, sentences, gold_upos_list, gold_deprel_list, gold_head_list,
                      model, tokenizer, device=None, max_sentences=None, desc=None):
    """
    Passa as embeddings APENAS pelos blocos indicados (0-based: bloco 11 = camada 12)
    e avalia os classificadores sobre a saída. blocks=[] avalia as embeddings puras.
    """
    if device is None:
        device = next(model.parameters()).device
    model.eval()

    encoder_layers = model.bert.encoder.layer
    total = upos_c = deprel_c = uas_c = las_c = 0

    n = len(sentences) if max_sentences is None else min(max_sentences, len(sentences))
    for i in tqdm(range(n), desc=desc):
        tokens = sentences[i]
        golds = {
            'upos':   np.array([UPOS_LABELS_TO_IDX[u]   for u in gold_upos_list[i]], dtype=np.int64),
            'deprel': np.array([DEPREL_LABELS_TO_IDX[d] for d in gold_deprel_list[i]], dtype=np.int64),
            'head':   np.array([int(h)                  for h in gold_head_list[i]], dtype=np.int64),
        }

        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                           padding=True, truncation=True).to(device)
        with torch.no_grad():
            # embeddings (posição + token type + LayerNorm), sem passar pelo encoder
            hs = model.bert.embeddings(
                input_ids=inputs['input_ids'],
                token_type_ids=inputs.get('token_type_ids'),
            )
            # batch=1 sem padding → máscara de atenção desnecessária
            for b in blocks:
                hs = encoder_layers[b](hs)
                if isinstance(hs, tuple):  # compatibilidade com versões antigas
                    hs = hs[0]

            word_ids = inputs.word_ids(batch_index=0)
            first_subs, tok_idxs = [], []
            seen = set()
            for pos, w in enumerate(word_ids):
                if w is not None and w not in seen and w < len(tokens):
                    seen.add(w)
                    first_subs.append(pos)
                    tok_idxs.append(w)
            if not first_subs:
                continue

            hs_tok = hs[0, torch.tensor(first_subs, device=device)]  # [n_tok, hidden]
            p_upos   = model.upos_classifier(hs_tok).argmax(-1).cpu().numpy()
            p_deprel = model.deprel_classifier(hs_tok).argmax(-1).cpu().numpy()
            p_head   = model.head_classifier(hs_tok).argmax(-1).cpu().numpy()

        g_upos, g_deprel, g_head = (golds[t][tok_idxs] for t in ('upos', 'deprel', 'head'))
        total    += len(tok_idxs)
        upos_c   += (p_upos == g_upos).sum()
        deprel_c += (p_deprel == g_deprel).sum()
        head_ok   = (p_head == g_head)
        uas_c    += head_ok.sum()
        las_c    += (head_ok & (p_deprel == g_deprel)).sum()

    blocos_str = ','.join(str(b + 1) for b in blocks) if blocks else 'nenhum (embeddings)'
    return {
        'blocos_ativos': blocos_str,
        'n_blocos': len(blocks),
        'upos_acc':   upos_c / total,
        'deprel_acc': deprel_c / total,
        'uas':        uas_c / total,
        'las':        las_c / total,
    }


In [ ]:
# ── Configurações: quais blocos ficam ativos (0-based; bloco 11 = camada 12) ───
block_configs = {
    'sanity: todos os blocos (== inferência normal)': list(range(12)),
    'APENAS bloco 12':                                [11],
    'APENAS bloco 12, aplicado 12x':                  [11] * 12,
    'blocos 11 e 12':                                 [10, 11],
    'blocos 9 a 12':                                  [8, 9, 10, 11],
    'blocos 7 a 12 (pula metade inferior)':           [6, 7, 8, 9, 10, 11],
    'nenhum bloco (embeddings puras)':                [],
}

block_rows = []
for name, blocks in block_configs.items():
    r = eval_block_subset(blocks, test_sentences, test_upos, test_deprel, test_head,
                          layerwise_model, layerwise_tokenizer, desc=name)
    r['config'] = name
    block_rows.append(r)

block_comparison = pd.DataFrame(block_rows)[
    ['config', 'blocos_ativos', 'n_blocos', 'upos_acc', 'deprel_acc', 'uas', 'las']
]
block_comparison.to_csv('./layerwise_block_skip_comparison.csv', index=False)
block_comparison.round(4)
